In [ ]:
import torch
import torch.nn as nn
from fla.layers import GatedDeltaNet
print(GatedDeltaNet)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.
d:\xianyu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
torch.compile is not available in Python 3.10, using identity decorator instead


<class 'fla.layers.gated_deltanet.GatedDeltaNet'>


In [ ]:
import torch
import torch.nn as nn

def build_model(num_classes = 7):
    model = GatedDeltaNet(num_classes).to(device)
    return model

class PatchEmbed(nn.Module):
    """
    图像分块嵌入 + 可学习位置编码
    输入: [B, 3, 224, 224]
    输出: [B, L, d_model]  (L=序列长度, d_model=模型维度)
    """
    def __init__(
        self,
        img_size=224,        
        patch_size=16,       
        in_chans=3,          
        d_model=512          
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size//patch_size) ** 2
        self.proj = nn.Conv2d(
            in_chans, d_model,
            kernel_size=patch_size,
            stride=patch_size,
            bias=False  
        )
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches, d_model)
        )
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        x = x + self.pos_embed
        return x

In [ ]:
class GDN(nn.Module):
    def __init__(self,in_channels,img_size,
                 patch_size,d_model,num_layers):
        super(GDN,self).__init__()
        self.in_channels,self.img_size,self.patch_size,self.d_model=in_channels,img_size,patch_size,d_model
        self.patchembed = PatchEmbed(img_size=img_size,patch_size=patch_size,in_channels=in_channels,embed_dim=d_model)
        self.modules = nn.ModuleList([build_model(num_claases = 7) for i in range(num_layers)])
        self.dropout = nn.Dropout(0.03)
    
    def forward(self,x):
        x = self.patchembed(x)
        x = self.dropout(x)
        for module in self.modules:
            x = module(x)
        return x
    
